# 02. EDA & Segmentation


In [34]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [35]:
# --- 0. COLOR PALETTE DEFINITION ---
c = {
    'white':        '#FFFFFF',
    'black':        '#000000',

    'blue_dark':    '#0F2854',
    'blue_medium':  '#1C4D8D',
    'blue_light':   '#4988C4',
    'blue_pale':    '#BDE8F5',

    'gray_light':   '#E5E5E5',
    'gray_medium':  '#CBCBCB',
    'gray_dark':    '#404040',

    'accent_gold':  '#E2A16F'
}
# --- 1. CORE STYLE FUNCTION ---

def apply_my_style(fig, title, subtitle=""):
    """
    Hàm áp dụng style chuẩn cho toàn bộ biểu đồ.
    """
    fig.update_layout(
        title={
            'text': f"<b style='font-size:26px; color:black;'>{title}</b><br>"
                    f"<span style='font-size:16px; color:#666;'>{subtitle}</span>",
            'x': 0.02,
            'y': 0.90,
        
            'xanchor': 'left',
            'yanchor': 'top',
            'font': {'family': "Montserrat, sans-serif"}
        },

        margin=dict(t=100, l=80, r=40, b=80),

        font=dict(family="Montserrat, sans-serif", size=14),
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99), # Legend gọn gàng

                # mirror=False removes the top and right border lines
        xaxis=dict(
            showgrid=False, 
            linecolor='#2C3E50', 
            linewidth=1, 
            mirror=False, 
            showline=True
        ),
        yaxis=dict(
            showgrid=False, 
            linecolor='#2C3E50', 
            linewidth=1, 
            mirror=False, 
            showline=True
        ),
    )
    
    
    return fig

In [36]:
data_path = '../Data/Processed/'

transactions_master = pd.read_parquet(f'{data_path}transactions_master.parquet')
demographics = pd.read_parquet(f'{data_path}demographics_imputed.parquet')
campaign_table = pd.read_parquet(f'{data_path}campaign_table_clean.parquet')
campaign_desc = pd.read_parquet(f'{data_path}campaign_desc_clean.parquet')
coupon = pd.read_parquet(f'{data_path}coupon_clean.parquet')
coupon_redempt = pd.read_parquet(f'{data_path}coupon_redempt_clean.parquet')
customer_base = pd.read_parquet(f'{data_path}customer_base_labeled.parquet')

print('Successfully loaded parquet datasets!')

Successfully loaded parquet datasets!


In [89]:
campaign_table.head()

,DESCRIPTION,household_key,CAMPAIGN
0,TypeA,17,26
1,TypeA,27,26
2,TypeA,212,26
3,TypeA,208,26
4,TypeA,192,26


In [90]:
campaign_desc.head()

,DESCRIPTION,CAMPAIGN,START_DAY,END_DAY
0,TypeB,24,659,719
1,TypeC,15,547,708
2,TypeB,25,659,691
3,TypeC,20,615,685
4,TypeB,23,646,684


In [91]:
coupon.head()

,COUPON_UPC,PRODUCT_ID,CAMPAIGN
0,10000089061,27160,4
1,10000089064,27754,9
2,10000089073,28897,12
3,51800009050,28919,28
4,52100000076,28929,25


In [92]:
coupon_redempt.head()

,household_key,DAY,COUPON_UPC,CAMPAIGN
0,1,421,10000085364,8
1,1,421,51700010076,8
2,1,427,54200000033,8
3,1,597,10000085476,18
4,1,597,54200029176,18


In [37]:
# --- 2. DATA PREPROCESSING ---

# Create a mapping for years based on 12-month cycles
# Year 1 = X, Year 2 = Y, Year 3 = Z...
year_naming = {0: 'X', 1: 'Y', 2: 'Z'}

# Calculate the Global Month Index (starting from 0)
# Every 4 weeks is considered 1 month
transactions_master['global_month_idx'] = (transactions_master['WEEK_NO'] - 1) // 4

# Calculate Year Offset and Month within that year
transactions_master['year_offset'] = transactions_master['global_month_idx'] // 12
transactions_master['month_in_year'] = (transactions_master['global_month_idx'] % 12) + 1

# Map the year offset to a letter (X, Y, Z)
transactions_master['year_letter'] = transactions_master['year_offset'].map(year_naming)

# Create the final display label (e.g., '1-X', '12-X', '1-Y')
transactions_master['date_label'] = (
    transactions_master['month_in_year'].astype(str) + '-' + transactions_master['year_letter']
)

# Aggregate revenue by global index and label to maintain chronological order
monthly_revenue = transactions_master.groupby(['global_month_idx', 'date_label'])['SALES_VALUE'].sum().reset_index()

# Ensure data is sorted by time before plotting
monthly_revenue = monthly_revenue.sort_values('global_month_idx')

In [38]:
churn_data = customer_base['is_churn'].value_counts(normalize=True).reset_index()
churn_data.columns = ['Status', 'Percentage']
churn_data['Percentage'] = churn_data['Percentage'] * 100
churn_data['Status'] = churn_data['Status'].map({0: 'Active', 1: 'Churned'})

fig = px.bar(
    churn_data,
    x='Status',
    y='Percentage',
    color='Status',
    text='Percentage',
    color_discrete_map={'Active': c['blue_medium'], 'Churned': c['accent_gold']}
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    marker_cornerradius=10,
    marker_line_width=0
)

fig = apply_my_style(
    fig,
    title="OVERALL CHURN PERCENTAGE",
    subtitle="Comparison of Active vs. Churned customers across the base"
)

fig.update_layout(
    xaxis_title="Customer Status",
    yaxis_title="Percentage (%)",
    yaxis=dict(range=[0, 100], showgrid=True, gridcolor=c['gray_light']),
    showlegend=False,
    width=700,
    height=600,
    bargap=0.5
)

fig.show()

# PART 1: DEMOGRAPHICS

In [39]:
demographics['household_key'].nunique(), transactions_master['household_key'].nunique()

(801, 2500)

The dataset includes 2,500 customers in total, but only about 801 customers have demographic information available; this could be a churn signal because they may not be willing to provide that information.

In [73]:
churned_customers = customer_base[customer_base['is_churn'] == 1].copy()
features_to_plot = ['AGE_DESC', 'INCOME_DESC', 'HOMEOWNER_DESC', 'MARITAL_STATUS_CODE', 'HH_COMP_DESC', 'KID_CATEGORY_DESC']
donut_palette = [c['blue_dark'], c['blue_medium'], c['blue_light'], c['accent_gold'], c['gray_medium'], c['blue_pale']]

fig = make_subplots(
    rows=2, cols=3, 
    specs=[[{'type': 'domain'}]*3]*2,
    subplot_titles=[f"<b>{feat.replace('_', ' ')}</b>" for feat in features_to_plot],
    horizontal_spacing=0.15,
    vertical_spacing=0.2
)

for i, feature in enumerate(features_to_plot):
    data = churned_customers[churned_customers[feature] != 'Unknown'][feature].value_counts()
    row = (i // 3) + 1
    col = (i % 3) + 1
    
    legend_key = f'legend{i+1}'
    
    fig.add_trace(
        go.Pie(
            labels=data.index,
            values=data.values,
            name=feature,
            hole=0.5,
            marker=dict(colors=donut_palette, line=dict(color=c['white'], width=2)),
            textinfo='percent',
            hoverinfo='label+value+percent',
            legend=legend_key
        ),
        row=row, col=col
    )

    x_pos = [0.22, 0.61, 1][col-1]
    y_pos = [0.82, 0.22][row-1]
    


    fig.update_layout({
        legend_key: dict(
            x=x_pos,
            y=y_pos,
            xanchor='left',
            yanchor='middle',
            font=dict(size=9),
            bgcolor='rgba(0,0,0,0)'
        )
    })

fig = apply_my_style(
    fig, 
    title="DEMOGRAPHIC COMPOSITION OF CHURNED CUSTOMERS", 
    subtitle="Analyzing the internal profile of households that have left the system (is_churn = 1)"
)

fig.update_layout(
    height=950,
    width=1600,
    margin=dict(t=180, l=50, r=150, b=50),
    showlegend=True
)

for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=16, family="Montserrat, sans-serif", color=c['blue_dark'])
    annotation['y'] = annotation['y'] + 0.03 

fig.show()

In [42]:
customer_base['Churn_Status'] = customer_base['is_churn'].map({0: 'Active', 1: 'Churned'})

for feature in features_to_plot:
    df_filtered = customer_base[customer_base[feature] != 'Unknown'].copy()

    composition_df = pd.crosstab(df_filtered['Churn_Status'], df_filtered[feature], normalize='index') * 100
    
    comp_melted = composition_df.reset_index().melt(
        id_vars='Churn_Status', 
        var_name=feature, 
        value_name='percentage'
    )

    fig = px.bar(
        comp_melted,
        x='Churn_Status',
        y='percentage',
        color=feature,
        text='percentage',
        color_discrete_sequence=px.colors.qualitative.T10 
    )

    fig.update_traces(
        texttemplate='%{text:.1f}%', 
        textposition='inside',
        insidetextanchor='middle',
        marker_line_color='white',
        marker_line_width=1,
        marker_cornerradius=8 
    )

    fig = apply_my_style(
        fig, 
        title=f"CHURN PROFILE BY {feature.replace('_', ' ')} (Excluding Unknowns)", 
        subtitle=f"Internal demographic breakdown of confirmed {feature} groups"
    )

    fig.update_layout(
        width=1600,     
        height=600,     
        bargap=0.7,     
        xaxis_title="Customer Status",
        yaxis_title="Composition Percentage (%)",
        barmode='stack',
        xaxis=dict(showgrid=False),
        yaxis=dict(range=[0, 100], showgrid=True, gridcolor=c['gray_light'])
    )

    fig.show()

In [64]:

for feature in features_to_plot:
    composition_df = pd.crosstab(customer_base[feature], customer_base['is_churn'], normalize='index') * 100

    comp_melted = composition_df.reset_index().melt(
        id_vars=feature, 
        var_name='Churn_Status', 
        value_name='percentage'
    )

    fig = px.bar(
        comp_melted,
        x=feature,
        y='percentage',
        color='Churn_Status',
        text='percentage',
    )

    fig.update_traces(
        texttemplate='%{text:.1f}%', 
        textposition='inside',
        insidetextanchor='middle',
        marker_line_color='white',
        marker_line_width=1,
        marker_cornerradius=8 
    )

    fig = apply_my_style(
        fig, 
        title=f"CHURN RATE BY {feature.replace('_', ' ')}", 
        subtitle=f"Analyzing the percentage of Active vs. Churned customers within each demographic segment"
    )

    fig.update_layout(
        width=1600,     
        height=650,     
        bargap=0.3,     
        
        xaxis_title=f"{feature.replace('_', ' ')} Group",
        yaxis_title="Percentage (%)",
        barmode='stack',
        legend_title_text="Status",
        xaxis=dict(showgrid=False),
        yaxis=dict(range=[0, 100], showgrid=True, gridcolor=c['gray_light'])
    )

    fig.show()

## INSIGHT ABOUT INCOME AND AGE

- Primary churn segments: Low-income customers (15–24K) and senior customers (65+).
- Most loyal segments: Young adult customers (25–34) and high-income customers (>200K).
- Core issue: The supermarket is gradually losing price-sensitive customers (low-income group) and a distinct customer segment (elderly customers).

In [78]:
transactions_customer_base = transactions_master.merge(customer_base, on='household_key', how='left')

In [87]:
# 2. Phân loại nhóm nghiên cứu: 15-24K vs Others
transactions_customer_base['income_group'] = transactions_customer_base['INCOME_DESC'].apply(
    lambda x: 'Target_15_24K' if x == '15-24K' else 'Other_Incomes'
)

# 3. Tính toán Basket Size (Số món hàng trung bình mỗi hóa đơn)
# Gom nhóm theo từng hóa đơn (BASKET_ID) để tính quy mô giỏ hàng
basket_metrics = transactions_customer_base.groupby(['income_group', 'is_churn', 'BASKET_ID']).agg({
    'QUANTITY': 'sum',
    'SALES_VALUE': 'sum',
    'COUPON_DISC': lambda x: 1 if (x < 0).any() else 0 # Đánh dấu hóa đơn có dùng coupon
}).reset_index()

# 4. Tính toán kết quả chẩn đoán cuối cùng
final_diagnostic = basket_metrics.groupby(['income_group', 'is_churn']).agg(
    avg_basket_size=('QUANTITY', 'mean'),
    avg_spend_per_basket=('SALES_VALUE', 'mean'),
    coupon_use_rate=('COUPON_DISC', 'mean') # Tỷ lệ % hóa đơn có sử dụng coupon
).reset_index()

# --- XUẤT KẾ QUẢ DƯỚI DẠNG TEXT ĐỂ KIỂM TRA ---
for index, row in final_diagnostic.iterrows():
    status = "CHURNED" if row['is_churn'] == 1 else "ACTIVE"
    print(f"Nhóm: {row['income_group']} | Trạng thái: {status}")
    print(f" - Số món trung bình/giỏ hàng: {row['avg_basket_size']:.2f}")
    print(f" - Số tiền trung bình/giỏ hàng: ${row['avg_spend_per_basket']:.2f}")
    print(f" - Tỷ lệ đơn hàng dùng Coupon: {row['coupon_use_rate']*100:.2f}%")
    print("-" * 50)

Nhóm: Other_Incomes | Trạng thái: ACTIVE
 - Số món trung bình/giỏ hàng: 13.56
 - Số tiền trung bình/giỏ hàng: $30.08
 - Tỷ lệ đơn hàng dùng Coupon: 5.66%
--------------------------------------------------
Nhóm: Other_Incomes | Trạng thái: CHURNED
 - Số món trung bình/giỏ hàng: 11.80
 - Số tiền trung bình/giỏ hàng: $25.92
 - Tỷ lệ đơn hàng dùng Coupon: 4.35%
--------------------------------------------------
Nhóm: Target_15_24K | Trạng thái: ACTIVE
 - Số món trung bình/giỏ hàng: 11.84
 - Số tiền trung bình/giỏ hàng: $24.42
 - Tỷ lệ đơn hàng dùng Coupon: 5.48%
--------------------------------------------------
Nhóm: Target_15_24K | Trạng thái: CHURNED
 - Số món trung bình/giỏ hàng: 13.76
 - Số tiền trung bình/giỏ hàng: $28.49
 - Tỷ lệ đơn hàng dùng Coupon: 3.50%
--------------------------------------------------


In [101]:
transactions_master = pd.merge(
    transactions_master, 
    customer_base[['household_key', 'is_churn']], 
    on='household_key', 
    how='left'
)

In [107]:
customer_base.head()

,household_key,mean_IPT,std_IPT,last_purchase_day,personalized_threshold,recency,is_churn,Frequency,Monetary,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,Churn_Status
0,1,8.506494,4.581494,706,17.669482,5,0,83,4310.160156,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,Active
1,10,142.750000,159.639959,685,462.029919,26,0,9,234.339996,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Active
2,100,32.619048,24.499951,691,81.618950,20,0,23,1959.219971,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Active
3,1000,5.256000,5.900935,706,17.057870,5,0,130,3972.439941,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Active
4,1001,8.897436,13.774237,710,36.445910,1,0,90,4074.020020,45-54,U,50-74K,Homeowner,Unknown,1,None/Unknown,Active


In [108]:
customer_base[customer_base['is_churn'] == 1]['Monetary'].describe()

count      302.000000
mean      2366.431641
std       2460.737549
min          8.170000
25%        583.987518
50%       1733.585022
75%       3286.027527
max      18828.119141
Name: Monetary, dtype: float64

In [110]:
customer_base[customer_base['is_churn'] == 0]['Monetary'].describe()

count     2198.000000
mean      3038.489014
std       3025.327148
min         28.959999
25%        960.314987
50%       2083.459961
75%       4090.029907
max      26285.330078
Name: Monetary, dtype: float64

In [111]:
customer_base['Churn_Status'] = customer_base['is_churn'].map({0: 'Active', 1: 'Churned'})

fig = px.box(
    customer_base,
    x='Churn_Status',
    y='Monetary',
    color='Churn_Status',
    color_discrete_map={'Active': c['blue_medium'], 'Churned': c['accent_gold']},
    points='outliers'
)

fig.update_traces(marker_size=3, line_width=1.5)

fig = apply_my_style(
    fig, 
    title="MONETARY DISTRIBUTION BY CHURN STATUS", 
    subtitle="Comparing total spending (Monetary) between Active and Churned households"
)

fig.update_layout(
    xaxis_title="Customer Status",
    yaxis_title="Total Monetary Value ($)",
    showlegend=False,
    width=900,
    height=700,
    yaxis=dict(showgrid=True, gridcolor=c['gray_light'])
)

fig.show()

In [112]:
customer_base[customer_base['is_churn'] == 1]['Frequency'].describe()

count    302.000000
mean      90.625828
std       98.589702
min        1.000000
25%       28.000000
50%       63.500000
75%      121.500000
max      821.000000
Name: Frequency, dtype: float64

In [113]:
customer_base[customer_base['is_churn'] == 0]['Frequency'].describe()

count    2198.000000
mean      101.799363
std       105.531950
min         2.000000
25%        38.000000
50%        74.000000
75%       129.750000
max      1223.000000
Name: Frequency, dtype: float64

In [115]:

fig = px.box(
    customer_base,
    x='Churn_Status',
    y='Frequency',
    color='Churn_Status',
    color_discrete_map={'Active': c['blue_medium'], 'Churned': c['accent_gold']},
    points='outliers'
)

fig.update_traces(marker_size=3, line_width=1.5)

fig = apply_my_style(
    fig, 
    title="FREQUENCY DISTRIBUTION BY CHURN STATUS", 
    subtitle="Comparing total spending (Monetary) between Active and Churned households"
)

fig.update_layout(
    xaxis_title="Customer Status",
    yaxis_title="Total Monetary Value ($)",
    showlegend=False,
    width=900,
    height=700,
    yaxis=dict(showgrid=True, gridcolor=c['gray_light'])
)

fig.show()

In [117]:
transactions_master[transactions_master['is_churn'] == 1]['COMMODITY_DESC'].value_counts().head(10)

COMMODITY_DESC
SOFT DRINKS               12774
FLUID MILK PRODUCTS        8374
BAKED BREAD/BUNS/ROLLS     8022
CHEESE                     7121
BAG SNACKS                 6510
BEEF                       5105
FRZN MEAT/MEAT DINNERS     4954
FROZEN PIZZA               4588
SOUP                       4448
YOGURT                     3912
Name: count, dtype: int64

In [118]:
transactions_master[transactions_master['is_churn'] == 0]['COMMODITY_DESC'].value_counts().head(10)

COMMODITY_DESC
SOFT DRINKS               104435
FLUID MILK PRODUCTS        77120
BAKED BREAD/BUNS/ROLLS     75095
CHEESE                     67662
BAG SNACKS                 60167
FRZN MEAT/MEAT DINNERS     51022
BEEF                       43530
SOUP                       41649
YOGURT                     40752
FROZEN PIZZA               38704
Name: count, dtype: int64

**Product Categories Analysis:** The product categories (COMMODITY_DESC) show remarkably similar distribution patterns between churned and active customers. Both groups purchase from the same top product categories in nearly identical proportions, suggesting that product affinity alone is not a strong churn predictor. 

In [120]:
transactions_master.info()

<class 'pandas.DataFrame'>
RangeIndex: 2548770 entries, 0 to 2548769
Data columns (total 21 columns):
 #   Column             Dtype  
---  ------             -----  
 0   household_key      str    
 1   BASKET_ID          str    
 2   DAY                int16  
 3   PRODUCT_ID         str    
 4   QUANTITY           int32  
 5   SALES_VALUE        float32
 6   STORE_ID           str    
 7   RETAIL_DISC        float32
 8   TRANS_TIME         int16  
 9   WEEK_NO            int8   
 10  COUPON_DISC        float32
 11  COUPON_MATCH_DISC  float32
 12  DEPARTMENT         str    
 13  BRAND              str    
 14  COMMODITY_DESC     str    
 15  global_month_idx   int8   
 16  year_offset        int8   
 17  month_in_year      int8   
 18  year_letter        str    
 19  date_label         str    
 20  is_churn           int32  
dtypes: float32(4), int16(2), int32(2), int8(4), str(9)
memory usage: 393.0 MB


In [121]:
brand_data = transactions_master[['household_key', 'BRAND']].merge(
    customer_base[['household_key', 'is_churn']], 
    on='household_key'
)

brand_counts = brand_data.groupby(['is_churn', 'BRAND']).size().reset_index(name='count')
brand_counts['percentage'] = brand_counts.groupby('is_churn')['count'].transform(lambda x: (x / x.sum()) * 100)
brand_counts['Status'] = brand_counts['is_churn'].map({0: 'Active', 1: 'Churned'})

fig = px.bar(
    brand_counts,
    x='BRAND',
    y='percentage',
    color='Status',
    barmode='group',
    text='percentage',
    color_discrete_map={'Active': c['blue_medium'], 'Churned': c['accent_gold']}
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    marker_cornerradius=10,
    marker_line_width=0
)

fig = apply_my_style(
    fig, 
    title="BRAND PREFERENCE: NATIONAL VS. PRIVATE", 
    subtitle="Analyzing the share of National and Private brand purchases between Active and Churned customers"
)

fig.update_layout(
    xaxis_title="Brand Category",
    yaxis_title="Share of Purchases (%)",
    yaxis=dict(range=[0, 100], showgrid=True, gridcolor=c['gray_light']),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    width=1000,
    height=700,
    bargap=0.2,
    bargroupgap=0.1
)

fig.show()

In [129]:
transactions_master['is_coupon_used'] =( transactions_master['COUPON_DISC'] + transactions_master['COUPON_MATCH_DISC']).apply(lambda x: 1 if x < 0 else 0)

In [130]:
transactions_master['is_coupon_used'] = (transactions_master['COUPON_DISC'] + transactions_master['COUPON_MATCH_DISC']).apply(lambda x: 1 if x < 0 else 0)

hh_coupon = transactions_master.groupby(['household_key', 'is_churn'])['is_coupon_used'].max().reset_index()
hh_analysis = pd.crosstab(hh_coupon['is_churn'], hh_coupon['is_coupon_used'], normalize='index') * 100
hh_analysis = hh_analysis.reset_index().melt(id_vars='is_churn', var_name='Coupon_User', value_name='Percentage')
hh_analysis = hh_analysis[hh_analysis['Coupon_User'] == 1]
hh_analysis['Status'] = hh_analysis['is_churn'].map({0: 'Active', 1: 'Churned'})

fig1 = px.bar(
    hh_analysis,
    x='Status',
    y='Percentage',
    color='Status',
    text='Percentage',
    color_discrete_map={'Active': c['blue_medium'], 'Churned': c['accent_gold']}
)

fig1.update_traces(texttemplate='%{text:.1f}%', textposition='outside', marker_cornerradius=10)
fig1 = apply_my_style(fig1, title="HOUSEHOLD COUPON ADOPTION RATE", subtitle="Percentage of households that have used at least one coupon")
fig1.update_layout(yaxis_title="Adoption Rate (%)", yaxis=dict(range=[0, 100]), showlegend=False, width=700, height=600)
fig1.show()

trans_penetration = transactions_master.groupby('is_churn')['is_coupon_used'].mean().reset_index()
trans_penetration['Percentage'] = trans_penetration['is_coupon_used'] * 100
trans_penetration['Status'] = trans_penetration['is_churn'].map({0: 'Active', 1: 'Churned'})

fig2 = px.bar(
    trans_penetration,
    x='Status',
    y='Percentage',
    color='Status',
    text='Percentage',
    color_discrete_map={'Active': c['blue_medium'], 'Churned': c['accent_gold']}
)

fig2.update_traces(texttemplate='%{text:.1f}%', textposition='outside', marker_cornerradius=10)
fig2 = apply_my_style(fig2, title="COUPON PENETRATION PER TRANSACTION", subtitle="Average percentage of baskets containing discounted items")
fig2.update_layout(yaxis_title="Penetration Rate (%)", yaxis=dict(range=[0, max(trans_penetration['Percentage'])*1.5]), showlegend=False, width=700, height=600)
fig2.show()

In [134]:
max_month = transactions_master['global_month_idx'].max()
last_4_months = transactions_master[transactions_master['global_month_idx'] > (max_month - 4)].copy()

monthly_stats = last_4_months.groupby(['global_month_idx', 'date_label', 'is_churn']).agg(
    total_baskets=('BASKET_ID', 'nunique'),
    total_households=('household_key', 'nunique')
).reset_index()

monthly_stats['avg_freq'] = monthly_stats['total_baskets'] / monthly_stats['total_households']
monthly_stats['Status'] = monthly_stats['is_churn'].map({0: 'Active', 1: 'Churned'})
monthly_stats = monthly_stats.sort_values('global_month_idx')

fig = px.line(
    monthly_stats,
    x='date_label',
    y='avg_freq',
    color='Status',
    markers=True,
    color_discrete_map={'Active': c['blue_medium'], 'Churned': c['accent_gold']}
)

fig.update_traces(line_width=4, marker_size=10)

fig = apply_my_style(
    fig, 
    title="VISIT FREQUENCY TREND (LAST 4 MONTHS)", 
    subtitle="Average number of transactions per household as they approach the churn point"
)

fig.update_layout(
    xaxis_title="Month-Year Cycle",
    yaxis_title="Avg Transactions per HH",
    width=1100,
    height=600
)

fig.show()